# Notebook 07 — Feature importance: what the Isolation Forest is actually paying attention to

The dashboard's SHAP panel answers *"why was this one cycle flagged?"*.
It doesn't answer the question an interviewer usually asks: **"what did
you find from feature importance, and why do those features matter?"**.

This notebook addresses that gap on FD001 and FD004:

1. Compute global SHAP across the full held-out test set (not just one
   engine).
2. Map the top sensors and feature families to engine subsystems and
   physical quantities using NASA's published sensor descriptions.
3. Check whether the same features matter in mid-life vs the warning
   zone (RUL ≤ 30).
4. Compare which sensors lead on FD001 (single regime, HPC fault) vs
   FD004 (six regimes, HPC + Fan faults) — do the multi-regime + multi-
   fault dynamics shift the importance signal?
5. Write findings in plain English so they're interview-ready.

The artefact you should leave with: a short table of top sensors with
physical meaning, and an explanation of why those sensors matter
mechanically — not just statistically.


## Setup


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_loader import load_cmapss, add_rul_to_train, create_anomaly_labels, get_sensor_columns
from src.preprocessing import train_test_split_by_unit
from src.feature_engineering import build_feature_pipeline
from src.multi_regime import apply_regime_normalisation
from src.models import IsolationForestDetector
from src.explainability import build_explainer, explain, FEATURE_FAMILIES
from src.sensor_descriptions import SENSOR_TABLE, INTERPRETATION, describe_sensor, subsystem_of

MODELS_ROOT = Path('../models')
SUBSET_REGIMES = {'FD001': 1, 'FD002': 6, 'FD003': 1, 'FD004': 6}

pd.set_option('display.max_colwidth', 80)


## Helper — recreate the held-out test set for a subset

Mirrors the dashboard's preprocessing path exactly so the SHAP values
we compute here are what the deployed model actually sees.


In [ ]:
def load_test_split(subset):
    train_df, _, _ = load_cmapss(subset)
    train_df = add_rul_to_train(train_df)
    train_df = create_anomaly_labels(train_df, threshold=30)
    sensor_cols = get_sensor_columns(train_df)
    train_df = train_df.astype({c: 'float64' for c in sensor_cols})

    subset_dir = MODELS_ROOT / subset
    with open(subset_dir / 'kept_sensors.json') as f:
        kept_sensors = json.load(f)

    if SUBSET_REGIMES[subset] > 1:
        km = joblib.load(subset_dir / 'kmeans.pkl')
        rs = joblib.load(subset_dir / 'regime_scalers.pkl')
        train_df = apply_regime_normalisation(train_df, sensor_cols, km, rs)

    dropped = set(sensor_cols) - set(kept_sensors)
    if dropped:
        train_df = train_df.drop(columns=list(dropped))

    feat = build_feature_pipeline(
        train_df, kept_sensors,
        rolling_windows=[5, 10], lags=[1, 5], ewma_spans=[5]
    )
    excl = {'unit_id', 'cycle', 'rul', 'anomaly', 'regime'}
    feature_cols = [c for c in feat.columns if c not in excl]

    sc = joblib.load(subset_dir / 'scaler.pkl')
    feat[feature_cols] = sc.transform(feat[feature_cols])

    _train_split, test_split = train_test_split_by_unit(feat, test_ratio=0.2, seed=42)

    iso = IsolationForestDetector()
    iso.load(str(subset_dir / 'isolation_forest.pkl'))
    return iso, test_split, feature_cols


## Compute global SHAP for FD001

A few thousand test rows is enough — `TreeExplainer` is exact, so
sample size only affects how much averaging we can do. We use the
healthy training rows as the background so SHAP values describe
deviations from "what healthy looks like".


In [ ]:
iso_001, test_001, feature_cols_001 = load_test_split('FD001')

# Background = healthy test rows (small, fast)
healthy = test_001[test_001['anomaly'] == 0]
X_bg = np.nan_to_num(healthy[feature_cols_001].sample(min(200, len(healthy)), random_state=42).values, nan=0.0)
X_test = np.nan_to_num(test_001[feature_cols_001].values, nan=0.0)
y_test = test_001['anomaly'].values
rul_test = test_001['rul'].values

print(f"Test set: {X_test.shape[0]} rows, {y_test.sum()} anomalous ({y_test.mean():.1%})")

explainer = build_explainer(iso_001, X_bg, max_background=200)
explanation = explain(iso_001, X_test, feature_cols_001, explainer=explainer)
print(f"SHAP values: {explanation.shap_values.shape}")


## Global top-20 features (raw names)

This is the table I'd start with in an interview. The y-axis is the
mean |SHAP| value across the whole test set — i.e., "how much does
this feature drive the IF's anomaly score, averaged across all cycles
of all 20 test engines".

Raw feature names are intentionally ugly. The next cell maps them to
plain English and groups by sensor.


In [ ]:
top20 = explanation.per_feature.head(20).copy()
top20


## Aggregate to top sensors with physical meaning

This is the table that actually answers "which features matter and
why". Each row collapses all engineered features for one base sensor
(rolling stats, lags, EWMAs, skew, kurt) into a single |SHAP| score
and looks up the sensor's physical quantity from the C-MAPSS spec.


In [ ]:
sensor_imp = explanation.per_sensor.copy()
sensor_imp = sensor_imp[sensor_imp['base_sensor'].str.startswith('sensor_')]
sensor_imp = sensor_imp.merge(
    SENSOR_TABLE.reset_index().rename(columns={'sensor': 'base_sensor'}),
    on='base_sensor', how='left',
)
sensor_imp['rank'] = range(1, len(sensor_imp) + 1)
sensor_imp_top = sensor_imp[['rank', 'base_sensor', 'symbol', 'quantity', 'subsystem', 'total_abs_shap', 'n_features']].head(10)
sensor_imp_top


## Mechanistic interpretation of the top sensors

For each top sensor, the C-MAPSS literature has a one-line story for
what its drift physically means in a degrading engine. The dashboard's
`pretty_feature_label` already shows this kind of context per feature;
here we collect it per sensor.


In [ ]:
def interpretation_paragraph(sensor):
    if sensor in INTERPRETATION:
        return INTERPRETATION[sensor]
    return f"(no interpretation paragraph in src/sensor_descriptions.py)"

for _, row in sensor_imp_top.iterrows():
    s = row['base_sensor']
    print(f"#{row['rank']:>2}  {describe_sensor(s)}")
    print(f"     |SHAP| total: {row['total_abs_shap']:.3f}  "
          f"({row['n_features']} engineered features collapsed)")
    print(f"     {interpretation_paragraph(s)}")
    print()


## Subsystem grouping — which part of the engine matters most?

Collapsing further: group the per-sensor importance by **engine
subsystem** (HPC, LPT, Fan, etc.). This is the answer to the deeper
question "which part of the engine drives the anomaly signal?" — and
it's often more interesting than the per-sensor view because it lines
up with how mechanical engineers actually think about jet engines.


In [ ]:
subsystem_imp = (
    sensor_imp.groupby('subsystem')
    .agg(total_abs_shap=('total_abs_shap', 'sum'),
         n_sensors=('base_sensor', 'count'))
    .sort_values('total_abs_shap', ascending=False)
    .reset_index()
)
subsystem_imp['share_%'] = (
    100 * subsystem_imp['total_abs_shap'] / subsystem_imp['total_abs_shap'].sum()
).round(1)
subsystem_imp


## Feature family ranking — what *kind* of signal matters?

The other useful aggregation: which feature family does the IF rely
on most? If kurtosis dominates rolling-mean, the model is sensitive to
distributional spikes (tail behaviour) rather than slow drift; if EWMA
dominates lag, recent-weighted averages outperform raw history.


In [ ]:
family_imp = explanation.per_family.copy()
family_imp = family_imp[family_imp['family'].isin([f for f in FEATURE_FAMILIES if f != 'other'])]
family_imp = family_imp.sort_values('total_abs_shap', ascending=False).reset_index(drop=True)
family_imp['share_%'] = (
    100 * family_imp['total_abs_shap'] / family_imp['total_abs_shap'].sum()
).round(1)
family_imp


## Temporal pattern — do the same features matter near failure?

Split the test set into three RUL buckets and recompute per-sensor
importance in each. If certain features only "light up" in the warning
zone, that's a real finding — it tells you which sensors are *late
indicators* vs which give early warning.


In [ ]:
def bucket_rul(r):
    if r > 80:
        return 'mid-life (RUL > 80)'
    if r > 30:
        return 'pre-warning (30 < RUL <= 80)'
    return 'warning zone (RUL <= 30)'

bucket = pd.Series(rul_test).apply(bucket_rul)
per_sensor_by_bucket = []
for label, idx in bucket.groupby(bucket).groups.items():
    arr = np.abs(explanation.shap_values[list(idx)])
    mean_abs = arr.mean(axis=0)
    df_b = pd.DataFrame({'feature': feature_cols_001, 'abs_shap': mean_abs})
    df_b['base_sensor'] = df_b['feature'].apply(
        lambda f: f.split('_roll_')[0].split('_lag_')[0].split('_diff_')[0]
                  .split('_ewma_')[0].split('_skew_')[0].split('_kurt_')[0]
                  if '_' in f and 'sensor_' in f else f
    )
    s_b = df_b.groupby('base_sensor')['abs_shap'].sum().sort_values(ascending=False)
    per_sensor_by_bucket.append((label, s_b))

top_sensors_overall = sensor_imp.head(7)['base_sensor'].tolist()
bucket_df = pd.DataFrame({
    label: s.reindex(top_sensors_overall).fillna(0).values
    for label, s in per_sensor_by_bucket
}, index=top_sensors_overall)
bucket_df = bucket_df[['mid-life (RUL > 80)', 'pre-warning (30 < RUL <= 80)', 'warning zone (RUL <= 30)']]
bucket_df.index = [describe_sensor(s).split('(')[1].split(' -')[0] for s in bucket_df.index]
bucket_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bucket_df.plot(kind='barh', ax=ax, width=0.8, color=['#4fc3f7', '#ffca28', '#ef5350'])
ax.set_xlabel('Total |SHAP| (sum of all engineered features for this sensor)')
ax.set_title('Top sensors\' importance across the engine\'s lifecycle')
ax.invert_yaxis()
ax.legend(title='Lifecycle stage', loc='lower right')
plt.tight_layout()
plt.savefig('../data/feature_importance_by_rul.png', dpi=150, bbox_inches='tight')
plt.show()


## Cross-subset: do FD004's top sensors differ from FD001's?

FD004 has six operating regimes and two fault modes — the multi-regime
+ multi-fault subset. If the same sensors dominate on both subsets,
the importance signal is robust; if FD004 shifts the ranking, that
tells us multi-regime data reveals different physical drivers.


In [ ]:
iso_004, test_004, feature_cols_004 = load_test_split('FD004')

X_bg_4 = np.nan_to_num(
    test_004[test_004['anomaly'] == 0][feature_cols_004]
    .sample(min(200, len(test_004)), random_state=42).values, nan=0.0
)
X_test_4 = np.nan_to_num(test_004[feature_cols_004].values, nan=0.0)
print(f"FD004 test set: {X_test_4.shape[0]} rows")

explainer_4 = build_explainer(iso_004, X_bg_4, max_background=200)
expl_4 = explain(iso_004, X_test_4, feature_cols_004, explainer=explainer_4)

sensor_imp_4 = expl_4.per_sensor.copy()
sensor_imp_4 = sensor_imp_4[sensor_imp_4['base_sensor'].str.startswith('sensor_')]
sensor_imp_4 = sensor_imp_4.merge(
    SENSOR_TABLE.reset_index().rename(columns={'sensor': 'base_sensor'}),
    on='base_sensor', how='left',
)


In [ ]:
# Side-by-side comparison
fd001_top10 = sensor_imp[['base_sensor', 'symbol', 'subsystem', 'total_abs_shap']].head(10).copy()
fd001_top10['FD001 rank'] = range(1, len(fd001_top10) + 1)
fd004_top10 = sensor_imp_4[['base_sensor', 'symbol', 'subsystem', 'total_abs_shap']].head(10).copy()
fd004_top10['FD004 rank'] = range(1, len(fd004_top10) + 1)

merged = fd001_top10.merge(
    fd004_top10, on=['base_sensor', 'symbol', 'subsystem'],
    suffixes=('_fd001', '_fd004'), how='outer',
)
merged['FD001 rank'] = merged['FD001 rank'].fillna('—')
merged['FD004 rank'] = merged['FD004 rank'].fillna('—')
merged = merged.sort_values('total_abs_shap_fd001', ascending=False, na_position='last')
merged[['base_sensor', 'symbol', 'subsystem', 'FD001 rank', 'FD004 rank', 'total_abs_shap_fd001', 'total_abs_shap_fd004']].head(15)


## Findings - what the IF actually pays attention to on FD001

Three findings from the cells above, all interview-ready and grounded
in the saved-model output (numbers are this run; deterministic for the
IF, so they reproduce).

### 1. The top sensor is downstream, not at the HPC

| Rank | Sensor | Subsystem | Total \|SHAP\| |
|---|---|---|---|
| 1 | P15 (bypass-duct total pressure) | Fan | 0.285 |
| 2 | NRc (corrected core speed) | Core | 0.236 |
| 3 | BPR (bypass ratio) | Performance | 0.225 |
| 4 | T50 (LPT outlet temperature) | LPT | 0.224 |
| 5 | W32 (LPT coolant bleed) | LPT | 0.218 |
| 6 | T30 (HPC outlet temperature) | HPC | 0.212 |
| 7 | phi (fuel-flow / Ps30) | Combustor | 0.207 |
| 8 | Ps30 (HPC outlet static pressure) | HPC | 0.207 |
| 9 | NRf (corrected fan speed) | Fan | 0.205 |
| 10 | htBleed (bleed enthalpy) | HPC | 0.200 |

The single most informative sensor isn't at the HPC where the fault
actually is - it's **P15, total pressure in the bypass duct** (a Fan-section
sensor). The model reads HPC degradation through the *airflow
redistribution* it causes: when the HPC loses efficiency, the
fan/core balance shifts and that shows up first in bypass-duct
pressure, before T30 or Ps30 (the direct HPC sensors) drift much.

**Aggregated by subsystem:** HPC takes 25.9% of total importance (4
sensors), Fan takes 21.8% (3 sensors). HPC still leads in aggregate,
but only because more sensors land in that group - not because each
HPC sensor individually dominates.

### 2. Drift slightly beats volatility, both beat distribution shape

Feature-family ranking (share of total \|SHAP\|):

| Family | Share | Mean signed SHAP | Interpretation |
|---|---|---|---|
| rolling mean (windowed average) | 19.8% | +0.180 | Slow drift, predominantly anomaly-pushing |
| rolling std (windowed volatility) | 16.6% | -0.010 | Noisiness rising, mixed direction |
| diff (k-cycle change) | 15.7% | -0.006 | Rate-of-change, mixed direction |
| lag (value k cycles ago) | 12.0% | +0.096 | History |
| EWMA (recent-weighted average) | 10.8% | +0.110 | Faster-reacting drift |
| raw (current sensor value) | 8.6% | +0.074 | Bottom of the pack |
| kurtosis (tail behaviour) | 8.6% | -0.023 | Spikes |
| skew (distribution asymmetry) | 7.9% | -0.012 | Asymmetry |

Two things to note: **engineered features clearly beat raw values**
(raw is bottom of the pack, validating the feature engineering work),
and **the rolling-mean signed-SHAP is strongly positive** (+0.18) while
the other families are near zero. That means rolling-mean features
are the most *directional* signal - when they fire, they push toward
anomaly; the others swing both ways.

### 3. Sharp early/late split across the lifecycle - the model uses different sensors at different times

Top sensors' \|SHAP\| in three RUL buckets:

| Sensor | mid-life (RUL > 80) | pre-warning (30 < RUL <= 80) | warning zone (RUL <= 30) |
|---|---|---|---|
| P15 (Fan) | **0.328** | 0.241 | 0.173 |
| NRc (Core) | 0.191 | 0.236 | **0.429** |
| BPR (Performance) | 0.190 | 0.210 | **0.398** |
| T50 (LPT) | 0.170 | 0.223 | **0.457** |
| W32 (LPT) | 0.193 | 0.200 | 0.353 |
| T30 (HPC) | 0.204 | 0.190 | 0.280 |
| phi (Combustor) | 0.181 | 0.202 | 0.330 |

This is the most interesting finding of the three:

- **P15 (the headline top sensor) is actually an *early* indicator** -
  its importance drops 47% from mid-life to the warning zone
  (0.328 → 0.173). It's strongest when the engine is healthy or
  early-degrading.
- **T50, NRc, BPR are *late* indicators** - importance rises 2-3x as
  failure approaches. T50 in particular goes 0.170 → 0.457 (2.7x).
- The IF is implicitly learning a *progression*: bypass-duct pressure
  drifts first, then LPT temperature + core speed + bypass ratio
  ramp up as the engine approaches failure.

This pattern is consistent with the physics: HPC efficiency loss
causes downstream temperature rises and forces the core to spin
faster to maintain thrust - both of which only become detectable as
the degradation accumulates. The bypass-duct pressure shift is more
subtle and dominates the signal while degradation is still mild.

### Cross-subset: FD004 changes which sensors carry the signal

| In FD001 top-10 only | In both | In FD004 top-10 only |
|---|---|---|
| P15, NRc, T30, phi, NRf | BPR, T50, W32, Ps30, htBleed | P30, farB, Nc, epr, Nf |

Pattern: FD001 favours "corrected" speeds (NRc, NRf - operating-
condition-normalised). FD004 favours their *physical* counterparts
(Nc, Nf - raw RPM) because per-regime normalisation already removes
the operating-condition variance, making the raw signal more
informative. Also, sensors that were constant on FD001 (epr, farB)
become informative on FD004 because they actually vary across the
six operating regimes.

### One paragraph for the interview

> "The headline finding is that the most important sensor on FD001
> isn't at the high-pressure compressor where the fault is - it's
> P15, the bypass-duct total pressure. The model picks up HPC
> degradation through the airflow redistribution it causes, not
> through direct compressor measurements. Even better: P15's
> importance actually decreases as failure approaches, while LPT
> temperature and core speed importance rises 2-3x in the warning
> zone. So the IF is implicitly learning a temporal progression -
> bypass-duct pressure drifts first, then LPT and core signals ramp
> up later. That's exactly the kind of mechanistic story the SHAP
> attribution gives you that pure F1 numbers can't."

That's a strong answer to the "what did you find from feature
importance?" question.
